In [1]:
# オリジナルデータをコピー

import os
import shutil

def copy_directory_with_metadata(src, dst):
    """
    Copies all contents of the source directory to the destination directory,
    preserving metadata (e.g., timestamps).
    """
    if not os.path.exists(src):
        raise FileNotFoundError(f"Source directory '{src}' does not exist.")
    
    if not os.path.exists(dst):
        os.makedirs(dst)
    
    for root, dirs, files in os.walk(src):
        # Calculate the relative path from the source to current root
        relative_path = os.path.relpath(root, src)
        dst_root = os.path.join(dst, relative_path)
        
        # Create directories in the destination
        for dir_name in dirs:
            os.makedirs(os.path.join(dst_root, dir_name), exist_ok=True)
        
        # Copy files to the destination
        for file_name in files:
            src_file = os.path.join(root, file_name)
            dst_file = os.path.join(dst_root, file_name)
            shutil.copy2(src_file, dst_file)  # Use copy2 to preserve metadata

# Paths
src_dir = "../mydata/00_original"
dst_dir = "../mydata/00_copy"

# Perform the copy
copy_directory_with_metadata(src_dir, dst_dir)

In [ ]:
# 不要なファイルの削除

import os

def remove_unwanted_files(root_dir, allowed_extensions):
    """
    Removes files that do not have one of the allowed extensions from the specified root directory.
    
    Parameters:
        root_dir (str): The root directory to process.
        allowed_extensions (set): A set of allowed file extensions (e.g., {".jpg", ".png"}).
    """
    removed_files = []
    for root, _, files in os.walk(root_dir):
        for file_name in files:
            file_path = os.path.join(root, file_name)
            _, ext = os.path.splitext(file_name)
            if ext.lower() not in allowed_extensions:
                os.remove(file_path)
                removed_files.append(file_path)
    
    # Print summary
    print(f"Removed {len(removed_files)} files:")
    for file in removed_files:
        print(file)

# Root directory
root_dir = "../mydata/00_copy"

# Allowed file extensions (case-insensitive)
allowed_extensions = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".gif", ".bmp", ".webp",
    ".doc", ".docx", ".ppt", ".pptx", ".xls", ".xlsx"
}

# Run the function
remove_unwanted_files(root_dir, allowed_extensions)

In [ ]:
# 空のディレクトリの削除

import os

def remove_empty_directories(root_dir):
    """
    Recursively removes empty directories from the specified root directory.
    
    Parameters:
        root_dir (str): The root directory to process.
    """
    removed_dirs = []
    
    for root, dirs, files in os.walk(root_dir, topdown=False):  # Start from the bottom of the tree
        for dir_name in dirs:
            dir_path = os.path.join(root, dir_name)
            # Check if the directory is empty
            if not os.listdir(dir_path):  # If the directory is empty
                os.rmdir(dir_path)
                removed_dirs.append(dir_path)
    
    # Print summary
    print(f"Removed {len(removed_dirs)} empty directories:")
    for dir_path in removed_dirs:
        print(dir_path)

# Root directory
root_dir = "../mydata/00_copy"

# Run the function
remove_empty_directories(root_dir)

In [ ]:
# ファイル名・フォルダ名の半角化
# ファイル名・フォルダ名からスペース除去

import os
import shutil
import unicodedata

def normalize_name(name):
    """
    Normalize a file or directory name by:
    - Converting full-width alphanumeric characters and symbols to half-width.
    - Removing both half-width and full-width spaces.
    """
    normalized = unicodedata.normalize("NFKC", name)
    normalized = normalized.replace(" ", "").replace("　", "")
    return normalized

def generate_unique_name(base_path, name, original_name):
    """
    Generate a unique name to prevent collisions by appending a counter (_1, _2, etc.).
    Only adds counter if the normalized name would cause a collision.
    """
    # まず正規化された名前が元の名前と同じかチェック
    if name == original_name:
        return name  # 変更がない場合はそのまま返す
    
    # 正規化された名前のパスが存在するかチェック
    new_path = os.path.join(base_path, name)
    if not os.path.exists(new_path):
        return name  # 重複がなければそのまま返す
    
    # 重複がある場合はカウンタを追加
    base_name, ext = os.path.splitext(name)
    counter = 1
    unique_name = name
    while os.path.exists(os.path.join(base_path, unique_name)):
        unique_name = f"{base_name}_{counter}{ext}"
        counter += 1
    return unique_name

def rename_files_and_directories(root_dir):
    """
    Recursively renames all files and directories under the root directory.
    - Converts full-width alphanumeric characters and symbols to half-width.
    - Removes half-width and full-width spaces.
    - Ensures unique names by appending a counter if collisions occur.
    """
    for root, dirs, files in os.walk(root_dir, topdown=False):
        # Rename files
        for file_name in files:
            old_path = os.path.join(root, file_name)
            new_name = normalize_name(file_name)
            
            # 正規化後の名前が元の名前と同じなら変更しない
            if new_name == file_name:
                continue
                
            new_name = generate_unique_name(root, new_name, file_name)
            new_path = os.path.join(root, new_name)
            
            if old_path != new_path:
                try:
                    shutil.move(old_path, new_path)
                    print(f"Renamed file: {old_path} -> {new_path}")
                except Exception as e:
                    print(f"Error renaming file {old_path}: {e}")
        
        # Rename directories
        for dir_name in dirs:
            old_path = os.path.join(root, dir_name)
            new_name = normalize_name(dir_name)
            
            # 正規化後の名前が元の名前と同じなら変更しない
            if new_name == dir_name:
                continue
                
            new_name = generate_unique_name(root, new_name, dir_name)
            new_path = os.path.join(root, new_name)
            
            if old_path != new_path:
                try:
                    shutil.move(old_path, new_path)
                    print(f"Renamed directory: {old_path} -> {new_path}")
                except Exception as e:
                    print(f"Error renaming directory {old_path}: {e}")

# Root directory
root_dir = "../mydata/00_copy"

# Run the function
rename_files_and_directories(root_dir)

In [11]:
import os
import shutil

def organize_standalone_files(target_dir, allowed_extensions=None):
    """
    指定されたディレクトリ直下にある単独ファイルを整理する関数
    
    Parameters:
    - target_dir: チェック対象のディレクトリパス
    - allowed_extensions: 処理対象とする拡張子のセット（Noneの場合はすべての拡張子を処理）
    """
    # 対象ディレクトリの存在確認
    if not os.path.exists(target_dir):
        print(f"エラー: 指定されたディレクトリ '{target_dir}' が存在しません。")
        return
    
    # 対象ディレクトリ直下のファイルを処理
    for item_name in os.listdir(target_dir):
        item_path = os.path.join(target_dir, item_name)
        
        # ファイルのみを処理対象とする
        if not os.path.isfile(item_path):
            continue
        
        # 拡張子の取得
        base_name, ext = os.path.splitext(item_name)
        
        # 拡張子フィルタリング（指定がある場合）
        if allowed_extensions and ext.lower() not in allowed_extensions:
            continue
        
        # 新しいディレクトリ名を作成（重複を避ける）
        new_dir_name = base_name
        new_dir_path = os.path.join(target_dir, new_dir_name)
        
        # 同名ディレクトリが既に存在する場合、カウンタを追加
        counter = 1
        while os.path.exists(new_dir_path):
            new_dir_name = f"{base_name}_{counter}"
            new_dir_path = os.path.join(target_dir, new_dir_name)
            counter += 1
        
        try:
            # 新しいディレクトリを作成
            os.makedirs(new_dir_path)
            print(f"ディレクトリを作成しました: {new_dir_path}")
            
            # ファイルを新しいディレクトリに移動
            new_file_path = os.path.join(new_dir_path, item_name)
            shutil.move(item_path, new_file_path)
            print(f"ファイルを移動しました: {item_path} -> {new_file_path}")
            
        except Exception as e:
            print(f"エラー: {item_path} の処理中に問題が発生しました: {e}")


    
# ユーザーが指定するチェック対象ディレクトリ

root_dir = os.path.join("../mydata/00_copy", "総合診療部")
target_directories = [os.path.join(root_dir, sub_d) for sub_d in os.listdir(root_dir)]

# オプション: 処理対象とする拡張子を指定
allowed_extensions = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".gif", ".bmp", ".webp",
    ".doc", ".docx", ".ppt", ".pptx", ".xls", ".xlsx", ".pdf"
}

# 関数実行
for target_dir in target_directories:
    organize_standalone_files(target_dir, allowed_extensions)

ディレクトリを作成しました: ../mydata/00_copy/総合診療部/わ行/渡辺ミチ子初診時
ファイルを移動しました: ../mydata/00_copy/総合診療部/わ行/渡辺ミチ子初診時.ppt -> ../mydata/00_copy/総合診療部/わ行/渡辺ミチ子初診時/渡辺ミチ子初診時.ppt
ディレクトリを作成しました: ../mydata/00_copy/総合診療部/わ行/渡辺益務
ファイルを移動しました: ../mydata/00_copy/総合診療部/わ行/渡辺益務.ppt -> ../mydata/00_copy/総合診療部/わ行/渡辺益務/渡辺益務.ppt
ディレクトリを作成しました: ../mydata/00_copy/総合診療部/わ行/Kr.渡辺綾子初診
ファイルを移動しました: ../mydata/00_copy/総合診療部/わ行/Kr.渡辺綾子初診.ppt -> ../mydata/00_copy/総合診療部/わ行/Kr.渡辺綾子初診/Kr.渡辺綾子初診.ppt
ディレクトリを作成しました: ../mydata/00_copy/総合診療部/は行/藤田ヨシ子
ファイルを移動しました: ../mydata/00_copy/総合診療部/は行/藤田ヨシ子.ppt -> ../mydata/00_copy/総合診療部/は行/藤田ヨシ子/藤田ヨシ子.ppt
ディレクトリを作成しました: ../mydata/00_copy/総合診療部/は行/藤沢正義
ファイルを移動しました: ../mydata/00_copy/総合診療部/は行/藤沢正義.ppt -> ../mydata/00_copy/総合診療部/は行/藤沢正義/藤沢正義.ppt
ディレクトリを作成しました: ../mydata/00_copy/総合診療部/は行/堀巌
ファイルを移動しました: ../mydata/00_copy/総合診療部/は行/堀巌.ppt -> ../mydata/00_copy/総合診療部/は行/堀巌/堀巌.ppt
ディレクトリを作成しました: ../mydata/00_copy/総合診療部/は行/広澤照子
ファイルを移動しました: ../mydata/00_copy/総合診療部/は行/広澤照子.ppt -> ../mydata/00_copy/総合診療部/は行/広澤